# Running `strata` as a server with a quantized model and multi-token-prediction

Now, we will try yet another software [`Strata`](https://github.com/Niko1221/Strata).
`Strata` can work with models larger than your GPU RAM and will stream n-gram
weights from the SSD. Experts are transfered to the GPU on demand from CPU RAM.

We will use the OpenAI client:

In [ ]:
from openai import OpenAI

An API key can be added, but we don't need it here

In [ ]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:9090/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

Start `Strata`with the command `$PWD/.venv/bin/python $PWD/serve/server.py --engine strata --config $PWD/strata-iq3_s.json --port 9090 --open`. The JSON file should contain:
```json
{
 "exe": "$STRATA/engine/strata",
 "args": [
  "--pack",
  "$DATA/packs/iq3_s",
  "--native",
  "$DATA/models/IQ3_S/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00001-of-00002.gguf",
  "--ple-gguf",
  "$DATA/models/IQ3_S/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00002-of-00002.gguf",
  "--expert-profile",
  "$STRATA/data/expert-profile.bin",
  "--expert-cache",
  "auto",
  "--prefill",
  "auto",
  "--spec",
  "4",
  "--spec-min-p",
  "0.5",
  "--mtp",
  "$DATA/mtp/rt",
  "--max-context",
  "131072",
  "--kv",
  "int8",
  "--ple-io",
  "ram",
  "--kv-resident",
  "32768",
  "--control-vector-scaled",
  "$STRATA/data/experimental-speed-projection/Qwen3.8-Flash-Next-experimental-speed-projection.gguf:1.0",
  "--control-vector-layer-range",
  "4",
  "44",
  "--cvec-mode",
  "project",
  "--cvec-dir",
  "per-layer",
  "--adapt-every",
  "1",
  "--adapt-swaps",
  "80",
  "--adapt-decay",
  "0.97"
 ],
 "cwd": "$STRATA",
 "tokenizer": "$DATA/packs/iq3_s/tokenizer",
 "model_name": "qwen3.8-flash-next-iq3_s",
 "log": "$STRATA/strata-iq3_s.log",
 "lib_dirs": [
  "/usr/local/cuda-13.3/bin",
  "/usr/local/cuda-13.3/lib64"
 ],
 "port": 9090,
 "gpu": 0,
 "gpus_asked": true,
 "host": "0.0.0.0",
 "user_args": [
  "--control-vector-scaled",
  "$STRATA/data/experimental-speed-projection/Qwen3.8-Flash-Next-experimental-speed-projection.gguf:1.0",
  "--control-vector-layer-range",
  "4",
  "44",
  "--cvec-mode",
  "project",
  "--cvec-dir",
  "per-layer",
  "--adapt-every",
  "1",
  "--adapt-swaps",
  "80",
  "--adapt-decay",
  "0.97"
 ],
 "args_written": [
  "--pack",
  "$DATA/packs/iq3_s",
  "--native",
  "$DATA/models/IQ3_S/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00001-of-00002.gguf",
  "--ple-gguf",
  "$DATA/models/IQ3_S/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00002-of-00002.gguf",
  "--expert-profile",
  "$STRATA/data/expert-profile.bin",
  "--expert-cache",
  "auto",
  "--prefill",
  "auto",
  "--spec",
  "4",
  "--spec-min-p",
  "0.5",
  "--mtp",
  "$DATA/mtp/rt",
  "--max-context",
  "131072",
  "--kv",
  "int8",
  "--ple-io",
  "ram",
  "--kv-resident",
  "32768",
  "--control-vector-scaled",
  "$STRATA/data/experimental-speed-projection/Qwen3.8-Flash-Next-experimental-speed-projection.gguf:1.0",
  "--control-vector-layer-range",
  "4",
  "44",
  "--cvec-mode",
  "project",
  "--cvec-dir",
  "per-layer",
  "--adapt-every",
  "1",
  "--adapt-swaps",
  "80",
  "--adapt-decay",
  "0.97"
 ]
}
```

Replace `$STRATA` and `$DATA` with the respective directories for your computer.

Compared to `llama.cpp`, the startup takes a bit longer. Let's see
if we can also gain additional speed with generation:

In [ ]:
model = "qwen3.8-flash-next-iq3_s"

In [ ]:
%%time
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "Erkläre den Heise Verlag!" } ]
)

Note the speed of the generation (in the terminal window). This is also very
fast, even if it cannot always reach the speed of `exl3`, especially with many
parallel requests.

In [ ]:
from IPython.display import display, Markdown
display(Markdown(completion.choices[0].message.content))

The `responses` API is also support by `ninfer`. Let's generate `python` code, which supposedly can also
be created by the draft model:

In [ ]:
%%time
response = client.responses.create(model=model, 
                                   input="Write a quicksort in Python")

In [ ]:
display(Markdown(response.output_text))

Now, take a look at the GPU memory usage.

In [ ]:
!nvidia-smi

`strata` uses almost the complete memory. Of course the model is also larger, but
still the speed is very good!

In [ ]:
%%time
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "Write a quicksort in Python!" } ]
)

In [ ]:
display(Markdown(response.output_text))